# Backend

> Backend selection

In [ ]:
#| default_exp backend

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export 

import os

In [ ]:
#| export

class BackendRegistry:
    """
    Registry of supported bioMONAI backends.

    The registry stores backend identifiers and optionally their metadata.
    It is used to validate the globally selected backend and to expose the
    available backends to users.
    """

    def __init__(self):
        self._backends = {}

    def register(self, name, **metadata):
        """
        Register a backend.

        Parameters
        ----------
        name : str
            Unique backend identifier.
        **metadata
            Optional backend metadata, such as its framework name or
            supported features.

        Returns
        -------
        str
            The normalized backend name.
        """
        name = self._normalize_name(name)

        if name in self._backends:
            raise ValueError(
                f"Backend '{name}' is already registered."
            )

        self._backends[name] = {
            "name": name,
            **metadata,
        }

        return name

    def unregister(self, name):
        """
        Remove a registered backend.
        """
        name = self._normalize_name(name)

        if name not in self._backends:
            raise ValueError(
                f"Backend '{name}' is not registered."
            )

        del self._backends[name]

    def get(self, name):
        """
        Return metadata for a registered backend.
        """
        name = self._normalize_name(name)

        try:
            return self._backends[name]
        except KeyError:
            raise ValueError(
                f"Unknown backend '{name}'. "
                f"Available backends: {self.names()}"
            ) from None

    def contains(self, name):
        """
        Return whether a backend is registered.
        """
        name = self._normalize_name(name)
        return name in self._backends

    def names(self):
        """
        Return registered backend names.
        """
        return tuple(self._backends)

    def __contains__(self, name):
        return self.contains(name)

    def __iter__(self):
        return iter(self._backends)

    def __len__(self):
        return len(self._backends)

    @staticmethod
    def _normalize_name(name):
        """
        Normalize and validate a backend name.
        """
        if not isinstance(name, str) or not name.strip():
            raise ValueError(
                "Backend name must be a non-empty string."
            )

        return name.strip().lower()

In [ ]:
#| export

BACKEND_REGISTRY = BackendRegistry()

In [ ]:
#| export

for _name, _metadata in {
    "monai": {
        "framework": "MONAI",
        "description": "MONAI/PyTorch backend",
    },
    "torch": {
        "framework": "PyTorch",
        "description": "Native PyTorch backend",
    },
    "fastai": {
        "framework": "fastai",
        "description": "fastai backend",
    },
    "keras": {
        "framework": "Keras",
        "description": "Keras backend",
    },
    "ignite": {
        "framework": "Ignite",
        "description": "PyTorch-Ignite backend",
    },
}.items():
    BACKEND_REGISTRY.register(_name, **_metadata)

del _name, _metadata

In [ ]:
#| export

_DEFAULT_BACKEND = "monai"
_ACTIVE_BACKEND = os.getenv(
    "BIOMONAI_BACKEND",
    _DEFAULT_BACKEND,
).strip().lower()


In [ ]:
#| export

if not BACKEND_REGISTRY.contains(_ACTIVE_BACKEND):
    raise ValueError(
        f"Unknown backend from BIOMONAI_BACKEND: "
        f"'{_ACTIVE_BACKEND}'. "
        f"Available backends: {BACKEND_REGISTRY.names()}"
    )

In [ ]:
#| export

def get_backend():
    """
    Return the currently active bioMONAI backend.

    Returns
    -------
    str
        Active backend identifier.
    """
    return _ACTIVE_BACKEND

In [ ]:
#| export

def set_backend(backend):
    """
    Set the active bioMONAI backend.

    Parameters
    ----------
    backend : str
        Backend identifier, such as ``"monai"``, ``"fastai"``,
        ``"torch"``, or ``"keras"``.

    Returns
    -------
    str
        The newly selected backend.

    Raises
    ------
    ValueError
        If the backend is unknown or invalid.
    """
    global _ACTIVE_BACKEND

    backend = BackendRegistry._normalize_name(backend)

    if not BACKEND_REGISTRY.contains(backend):
        available = ", ".join(BACKEND_REGISTRY.names())

        raise ValueError(
            f"Unknown backend '{backend}'. "
            f"Available backends: {available}"
        )

    _ACTIVE_BACKEND = backend

    return _ACTIVE_BACKEND

In [ ]:
#| export

def available_backends():
    """
    Return the registered bioMONAI backends.

    Returns
    -------
    tuple
        Registered backend identifiers.
    """
    return BACKEND_REGISTRY.names()

In [ ]:
#| export

def backend_info(backend=None):
    """
    Return metadata for a backend.

    Parameters
    ----------
    backend : str, optional
        Backend identifier. If ``None``, the active backend is used.
    """
    backend = get_backend() if backend is None else backend
    return BACKEND_REGISTRY.get(backend)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()